# Notebook 16 — 3B Ceiling Baseline · CommonsenseQA
## SLM-to-SLM Guided Reasoning — Compute Ceiling Comparison

**Purpose:** Run Qwen2.5-3B alone with 5 votes — no fine-tuning, no LoRA, no guide.
This is the **compute ceiling** (15B param-passes) vs our pipeline's 10.5B.

| Condition | Setup | Compute |
|-----------|-------|---------|
| Baseline | 1.5B × 5 | 7.5B |
| **Our Pipeline** | 3B guide (LoRA) + 1.5B × 5 | **10.5B** |
| **← This notebook** | 3B base × 5 | **15.0B** |

Same 900 CommonsenseQA questions · Same seed=42 · Same temp=0.4 · Same 3-angle evaluation

**Why this matters:** CommonsenseQA is the most important out-of-domain result — the guide was trained on math (GSM8K) and still helped. Does the raw 3B model, with no fine-tuning, do even better at higher compute?

In [2]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [3]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('HuggingFace login done')

HuggingFace login done


In [4]:
# CELL 3 -- Imports + GPU check
import os, json, re, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/csqa_ceiling"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"PyTorch : {torch.__version__}")
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory/1024**3:.1f} GB")
else:
    print("No GPU detected")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 15.9 GB
Output  : /kaggle/working/csqa_ceiling


In [5]:
# CELL 4 -- Configuration
# ONE model only. No guide. No LoRA. No fine-tuning.
# Ceiling condition: 3B x 5 votes = 15B param-passes.
CONFIG = {
    "model_name"          : "Qwen/Qwen2.5-3B-Instruct",
    "model_params_B"      : 3.0,
    # Dataset
    "dataset_name"        : "tau/commonsense_qa",
    "dataset_split"       : "validation",     # test labels withheld; validation has 1,221 Qs
    "max_eval_samples"    : 900,              # same as pipeline N=900 run
    "random_seed"         : 42,              # FIXED -- must match pipeline run exactly
    # Voting
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,             # same as pipeline N=900 run
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 400,
    # Random chance for CommonsenseQA (5 options A-E)
    "random_chance"       : 20.0,
    # Known results from N=900 pipeline run (for comparison in Angle 1)
    "known_baseline_acc"  : 70.7,            # 1.5B x 5 @ 7.5B
    "known_pipeline_acc"  : 75.8,            # 3B+LoRA + 1.5B x 5 @ 10.5B
    "known_baseline_B"    : 7.5,
    "known_pipeline_B"    : 10.5,
    # Output paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}
print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<26}: {v}")

Config ready:
  model_name                : Qwen/Qwen2.5-3B-Instruct
  model_params_B            : 3.0
  dataset_name              : tau/commonsense_qa
  dataset_split             : validation
  max_eval_samples          : 900
  random_seed               : 42
  n_votes                   : 5
  vote_temperature          : 0.4
  refiner_temperature       : 0.3
  max_new_tokens            : 400
  random_chance             : 20.0
  known_baseline_acc        : 70.7
  known_pipeline_acc        : 75.8
  known_baseline_B          : 7.5
  known_pipeline_B          : 10.5
  results_file              : /kaggle/working/csqa_ceiling/results.jsonl
  report_file               : /kaggle/working/csqa_ceiling/eval_report.json
  angle1_file               : /kaggle/working/csqa_ceiling/angle1_compute_efficiency.json
  angle2_file               : /kaggle/working/csqa_ceiling/angle2_vote_consistency.json
  angle3_file               : /kaggle/working/csqa_ceiling/angle3_confidence_calibration.json
  checkpoin

In [6]:
# CELL 5 -- Load CommonsenseQA dataset
# Fields: id, question, question_concept, choices (label + text), answerKey
# We format as: "<question>\n\nOptions:\nA) ...\nB) ...\nC) ...\nD) ...\nE) ..."
import random

VALID_LETTERS = set("ABCDE")

def normalise_csqa(item):
    labels      = item["choices"]["label"]    # ['A','B','C','D','E']
    texts       = item["choices"]["text"]     # list of option strings
    options_str = "\n".join(f"{l}) {t}" for l, t in zip(labels, texts))
    q   = item["question"].strip() + "\n\nOptions:\n" + options_str
    ans = str(item["answerKey"]).strip().upper()
    return {
        "question" : q,
        "answer"   : ans,
        "concept"  : item.get("question_concept", ""),
    }

print("Loading CommonsenseQA from HuggingFace...")
raw_ds   = load_dataset(CONFIG["dataset_name"])
all_data = [normalise_csqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Val size : {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Formatted: {len(all_data)} questions")

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
n = min(CONFIG["max_eval_samples"], len(all_data))
test_data = random.sample(all_data, n)

print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print(f"\nSample question:\n{test_data[0]['question']}")
print(f"Concept : {test_data[0]['concept']}")
print(f"Answer  : {test_data[0]['answer']}")

Loading CommonsenseQA from HuggingFace...
Splits   : ['train', 'validation', 'test']
Val size : 1221
Formatted: 1221 questions
Sampled 900 questions (seed=42)

Sample question:
You can do knitting to get the feeling of what?

Options:
A) relaxation
B) arthritis
C) adrenaline
D) your
E) sweater may produced
Concept : knitting
Answer  : A


In [7]:
# CELL 6 -- Answer extraction for MCQ (A-E)
# CommonsenseQA answers are single letters. Identical to pipeline notebook.

def extract_gt_answer(answer_str):
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""

def extract_pred_answer(text):
    text = text.strip()
    # 1. Conclusive phrases: "the answer is X"
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 2. "option/choice X is correct"
    m = re.search(
        r"(?:option|choice)\s+([A-E])\s+(?:is correct|is the answer|matches|is right|is most likely)",
        text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 3. #### A
    m = re.search(r"####\s*([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 4. (A) at end
    m = re.search(r"\(([A-E])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 5. **A**
    m = re.search(r"\*\*([A-E])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS: return m.group(1).upper()
    # 6. Standalone letter on its own line (last)
    matches = re.findall(r"^\s*([A-E])\s*$", text, re.MULTILINE | re.IGNORECASE)
    if matches: return matches[-1].upper()
    # 7. Last standalone letter anywhere
    matches = re.findall(r"\b([A-E])\b", text, re.IGNORECASE)
    if matches: return matches[-1].upper()
    return ""

# Self-test
_tests = [
    ("The answer is C",     "C"),
    ("the correct answer is B", "B"),
    ("After thinking, the answer is: D", "D"),
    ("#### E",              "E"),
    ("(A)",                 "A"),
    ("**B**",               "B"),
    ("random text",         ""),
]
ok = all(extract_pred_answer(t)==e for t,e in _tests)
print("Extractor:", "ALL PASSED" if ok else "FAILURES DETECTED")
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    print(f"  {'OK' if got==exp else 'FAIL'}  '{txt}' -> '{got}'")

Extractor: ALL PASSED
  OK  'The answer is C' -> 'C'
  OK  'the correct answer is B' -> 'B'
  OK  'After thinking, the answer is: D' -> 'D'
  OK  '#### E' -> 'E'
  OK  '(A)' -> 'A'
  OK  '**B**' -> 'B'
  OK  'random text' -> ''


In [8]:
# CELL 7 -- Load 3B model (BASE only -- NO LoRA, NO fine-tuning)
# Critical difference: no PeftModel, no adapter path, pure base weights.
print(f"Loading: {CONFIG['model_name']}")
print("Adapter: NONE -- base model only, no task-specific training")

model_tok = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Model VRAM : {used:.2f} GB / {total:.1f} GB")
    print(f"Headroom   : {total - used:.1f} GB")

print(f"Compute per question: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print("Model ready -- no fine-tuning, no adapter")

Loading: Qwen/Qwen2.5-3B-Instruct
Adapter: NONE -- base model only, no task-specific training


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model VRAM : 5.75 GB / 15.9 GB
Headroom   : 10.1 GB
Compute per question: 3.0B x 5 = 15.0B param-passes
Model ready -- no fine-tuning, no adapter


In [9]:
# CELL 8 -- Generation functions
# CommonsenseQA-specific system prompts for the base 3B solver.

SOLVE_SYSTEM = (
    "You are a commonsense question answering assistant.\n"
    "Read the question carefully. Use your general knowledge to pick the best answer.\n"
    "Think step by step if needed.\n"
    "No markdown. Plain text only.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful commonsense reasoning checker.\n"
    "You are given a question and a list of candidate answers that are tied.\n"
    "Reason step by step about which answer is most plausible given real-world knowledge.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

def run_3b(messages, max_tokens, temperature):
    prompt = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = model_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    dev    = next(model_3b.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = model_3b.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = model_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return model_tok.decode(new_toks, skip_special_tokens=True).strip()

def generate_solve(question):
    return run_3b(
        [{"role":"system","content":SOLVE_SYSTEM},
         {"role":"user",  "content":question}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )

def generate_refine(question, candidates):
    cands   = ", ".join(sorted(set(c for c in candidates if c)))
    content = f"{question}\n\nPrevious attempts gave different answers: {cands}\nRe-reason carefully and pick the single best letter:"
    return run_3b(
        [{"role":"system","content":REFINER_SYSTEM},
         {"role":"user",  "content":content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )

print("Generation functions ready")
print(f"  generate_solve()   -- 3B base, temp {CONFIG['vote_temperature']}, no plan")
print(f"  generate_refine()  -- 3B base, temp {CONFIG['refiner_temperature']}, tie-breaker")

Generation functions ready
  generate_solve()   -- 3B base, temp 0.4, no plan
  generate_refine()  -- 3B base, temp 0.3, tie-breaker


In [10]:
# CELL 9 -- Voting logic (identical to pipeline notebook)
def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])
    refiner_used = False; refiner_correct = None

    if is_majority:
        final = top_answer; strategy = "majority"
        conf  = round(top_count / total, 4); wasted = total - top_count
    else:
        ref_raw         = generate_refine(question, list(answers))
        ref_ans         = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_votes   = answers + [ref_ans]
        new_counts  = Counter(all_votes)
        new_common  = new_counts.most_common()
        new_top     = new_common[0][0]
        new_top_c   = new_common[0][1]
        still_tied  = len(new_common) > 1 and new_top_c == new_common[1][1]
        final       = new_top
        strategy    = "coin_flip" if still_tied else "refiner_tiebreak"
        conf        = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts; total = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"    : final, "strategy":strategy, "confidence":conf,
        "vote_counts"     : dict(vote_counts), "correct_votes":correct_votes,
        "total_votes"     : total, "vote_consistency":round(vote_consistency,4),
        "wasted_votes"    : wasted, "refiner_used":refiner_used,
        "refiner_correct" : refiner_correct,
    }

print("Voting logic ready (majority / refiner_tiebreak / coin_flip)")

Voting logic ready (majority / refiner_tiebreak / coin_flip)


In [11]:
# CELL 10 -- Single question test
print("=" * 65)
print("SINGLE QUESTION TEST  (CommonsenseQA -- 3B Ceiling)")
print("=" * 65)
item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question :\n{q}")
print(f"Concept  : {item['concept']}")
print(f"GT Answer: {gt}")
print(f"\nRunning {CONFIG['n_votes']} votes (3B base, no plan, temp={CONFIG['vote_temperature']})...")

votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_solve(q)
    pred = extract_pred_answer(raw)
    votes_raw.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

dec = vote_and_decide(votes_raw, q, gt)
print(f"\n  Result    : {dec['final_answer']}  (GT: {gt})  {'CORRECT' if dec['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {dec['strategy']}")
print(f"  Confidence: {dec['confidence']}")
print(f"  Correct votes: {dec['correct_votes']}/{dec['total_votes']}")
print(f"  Vote counts  : {dec['vote_counts']}")
print("\nTest done -- run Cell 11 for full 900-question evaluation")

SINGLE QUESTION TEST  (CommonsenseQA -- 3B Ceiling)
Question :
You can do knitting to get the feeling of what?

Options:
A) relaxation
B) arthritis
C) adrenaline
D) your
E) sweater may produced
Concept  : knitting
GT Answer: A

Running 5 votes (3B base, no plan, temp=0.4)...
  Vote 1: 'A'  |  raw[:80]: To determine the correct answer, let's consider each option in relation to knitt
  Vote 2: 'A'  |  raw[:80]: To determine the correct answer, let's consider each option in relation to the a
  Vote 3: 'A'  |  raw[:80]: To determine the correct answer, let's consider each option in relation to the a
  Vote 4: 'A'  |  raw[:80]: To determine the correct answer, let's consider each option in relation to the a
  Vote 5: 'A'  |  raw[:80]: To determine the correct answer, let's consider each option in relation to the a

  Result    : A  (GT: A)  CORRECT
  Strategy  : majority
  Confidence: 1.0
  Correct votes: 5/5
  Vote counts  : {'A': 5}

Test done -- run Cell 11 for full 900-question evaluati

In [12]:
# CELL 11 -- Full Evaluation Loop
# 900 questions, single ceiling_3b condition, with op_type stored per record.
# Checkpointing every save_every questions -- safe to interrupt and resume.

print(f"CommonsenseQA 3B Ceiling: {len(test_data)} questions")
print(f"Model  : {CONFIG['model_name']} (base, no LoRA)")
print(f"Votes  : {CONFIG['n_votes']} x temp {CONFIG['vote_temperature']}")
print(f"Compute: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print(f"Random chance: {CONFIG['random_chance']}% (5-option MCQ)")
print("-" * 65)

results   = []
start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from index {start_idx} ({len(results)} saved)")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="CSQA 3B Ceiling"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    try:
        votes_raw = [extract_pred_answer(generate_solve(question))
                     for _ in range(CONFIG["n_votes"])]
        dec = vote_and_decide(votes_raw, question, gt_answer)
        results.append({
            "mode"            : "ceiling_3b",
            "idx"             : idx,
            "question"        : question,
            "concept"         : item.get("concept", ""),
            "gt_answer"       : gt_answer,
            "final_answer"    : dec["final_answer"],
            "correct"         : dec["final_answer"] == gt_answer,
            "strategy"        : dec["strategy"],
            "confidence"      : dec["confidence"],
            "correct_votes"   : dec["correct_votes"],
            "total_votes"     : dec["total_votes"],
            "vote_consistency": dec["vote_consistency"],
            "wasted_votes"    : dec["wasted_votes"],
            "refiner_used"    : dec["refiner_used"],
            "refiner_correct" : dec["refiner_correct"],
            "vote_counts"     : dec["vote_counts"],
        })
    except RuntimeError as e:
        results.append({
            "mode":"ceiling_3b","idx":idx,"question":question,"concept":item.get("concept",""),
            "gt_answer":gt_answer,"final_answer":"","correct":False,"strategy":"error",
            "confidence":0.0,"correct_votes":0,"total_votes":CONFIG["n_votes"],
            "vote_consistency":0.0,"wasted_votes":CONFIG["n_votes"],
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"],"w") as f:
            for r in results: f.write(json.dumps(r)+"\n")
        with open(CONFIG["checkpoint_file"],"w") as f:
            json.dump({"last_index":idx+1},f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}/{len(test_data)}]  3B Ceiling: {acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"],"w") as f:
    for r in results: f.write(json.dumps(r)+"\n")
with open(CONFIG["checkpoint_file"],"w") as f:
    json.dump({"last_index":len(test_data)},f)

correct = sum(r["correct"] for r in results)
acc     = correct/len(results)*100
print(f"\nEvaluation complete.")
print(f"  3B Ceiling : {correct}/{len(results)} = {acc:.1f}%")
print(f"  Pipeline   : {CONFIG['known_pipeline_acc']}%  (known, 10.5B, N=900)")
print(f"  Baseline   : {CONFIG['known_baseline_acc']}%  (known, 7.5B, N=900)")

CommonsenseQA 3B Ceiling: 900 questions
Model  : Qwen/Qwen2.5-3B-Instruct (base, no LoRA)
Votes  : 5 x temp 0.4
Compute: 3.0B x 5 = 15.0B param-passes
Random chance: 20.0% (5-option MCQ)
-----------------------------------------------------------------
Resumed from index 150 (150 saved)


CSQA 3B Ceiling:   0%|          | 0/750 [00:00<?, ?it/s]

  [175/900]  3B Ceiling: 80.0%  (15.2 min)
  [200/900]  3B Ceiling: 78.0%  (31.7 min)
  [225/900]  3B Ceiling: 78.2%  (49.3 min)
  [250/900]  3B Ceiling: 78.8%  (67.9 min)
  [275/900]  3B Ceiling: 78.5%  (85.3 min)
  [300/900]  3B Ceiling: 79.0%  (100.4 min)
  [325/900]  3B Ceiling: 79.1%  (116.3 min)
  [350/900]  3B Ceiling: 77.7%  (134.4 min)
  [375/900]  3B Ceiling: 78.1%  (150.5 min)
  [400/900]  3B Ceiling: 78.2%  (165.5 min)
  [425/900]  3B Ceiling: 77.6%  (180.6 min)
  [450/900]  3B Ceiling: 78.2%  (197.4 min)
  [475/900]  3B Ceiling: 77.9%  (213.6 min)
  [500/900]  3B Ceiling: 77.0%  (228.9 min)
  [525/900]  3B Ceiling: 76.2%  (247.0 min)
  [550/900]  3B Ceiling: 76.0%  (263.9 min)
  [575/900]  3B Ceiling: 75.7%  (281.7 min)
  [600/900]  3B Ceiling: 75.8%  (297.0 min)
  [625/900]  3B Ceiling: 76.0%  (311.4 min)
  [650/900]  3B Ceiling: 75.4%  (327.7 min)
  [675/900]  3B Ceiling: 75.4%  (344.2 min)
  [700/900]  3B Ceiling: 75.6%  (361.6 min)
  [725/900]  3B Ceiling: 75.7%  (376.

In [13]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
ceiling_compute  = CONFIG["model_params_B"] * CONFIG["n_votes"]
pipeline_compute = CONFIG["known_pipeline_B"]
baseline_compute = CONFIG["known_baseline_B"]

ceiling_acc  = sum(r["correct"] for r in results)/len(results)*100
pipeline_acc = CONFIG["known_pipeline_acc"]
baseline_acc = CONFIG["known_baseline_acc"]
random_chance = CONFIG["random_chance"]

ceiling_eff  = ceiling_acc  / ceiling_compute
pipeline_eff = pipeline_acc / pipeline_compute
baseline_eff = baseline_acc / baseline_compute

n = len(results); N = CONFIG["n_votes"]
ceiling_wasted = sum(r["wasted_votes"] for r in results)
strategy_stats = {}
for r in results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

ref_triggered = sum(r["refiner_used"] for r in results)
ref_correct   = sum(1 for r in results if r["refiner_used"] and r.get("refiner_correct"))

print("=" * 68)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (CommonsenseQA -- Three-Way Comparison)")
print("=" * 68)
print(f"  Random chance baseline: {random_chance}% (5 options A-E)")
print()
print(f"  {'Setup':<38} | {'Compute':>8} | {'Accuracy':>9} | {'Above Chance':>13} | {'Acc/B':>7}")
print(f"  {'-'*38}-+-{'-'*8}-+-{'-'*9}-+-{'-'*13}-+-{'-'*7}")
print(f"  {'Baseline  (1.5B x 5)':<38} | {baseline_compute:>6.1f}B  | {baseline_acc:>8.1f}% | {baseline_acc-random_chance:>+12.1f}% | {baseline_eff:>6.3f}")
print(f"  {'Pipeline  (3B+LoRA x1 + 1.5B x5)':<38} | {pipeline_compute:>6.1f}B  | {pipeline_acc:>8.1f}% | {pipeline_acc-random_chance:>+12.1f}% | {pipeline_eff:>6.3f}")
print(f"  {'Ceiling   (3B base x 5)  <- THIS':<38} | {ceiling_compute:>6.1f}B  | {ceiling_acc:>8.1f}% | {ceiling_acc-random_chance:>+12.1f}% | {ceiling_eff:>6.3f}")
print()
print(f"  Pipeline vs Ceiling  : {pipeline_acc - ceiling_acc:+.1f} pts  (pipeline uses {ceiling_compute - pipeline_compute:.1f}B LESS)")
print(f"  Ceiling  vs Baseline : {ceiling_acc  - baseline_acc:+.1f} pts  (+{ceiling_compute - baseline_compute:.1f}B more)")
print(f"  Pipeline vs Baseline : {pipeline_acc - baseline_acc:+.1f} pts  (+{pipeline_compute - baseline_compute:.1f}B more)")
print()
if pipeline_acc >= ceiling_acc:
    print(f"  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.")
    print(f"  Structured verbal planning outperforms raw model capacity on commonsense reasoning.")
else:
    gap  = ceiling_acc - pipeline_acc
    frac = (pipeline_acc - baseline_acc) / max(ceiling_acc - baseline_acc, 0.01) * 100
    print(f"  RESULT: 3B ceiling leads pipeline by {gap:.1f} pts at 43% higher cost.")
    print(f"  Pipeline recovers {frac:.0f}% of ceiling gain at {pipeline_compute/ceiling_compute*100:.0f}% of ceiling cost.")
    print(f"  Note: guide was trained on math (GSM8K), not commonsense.")
    print(f"  A multi-domain fine-tuned guide may close this gap.")

print(f"\n  Wasted votes -- Ceiling: {ceiling_wasted}/{n*N} ({ceiling_wasted/(n*N)*100:.1f}%)")
if ref_triggered > 0:
    print(f"  Refiner: triggered {ref_triggered}, correct {ref_correct}")
print(f"\n  Strategy breakdown:")
for s,v in sorted(strategy_stats.items(), key=lambda x:-x[1]['n']):
    acc_s = v['correct']/v['n']*100 if v['n'] else 0
    print(f"    {s:<22}  n={v['n']:4d}  acc={acc_s:.1f}%")

angle1 = {
    "dataset":"CommonsenseQA","experiment":"ceiling_3b","n_questions":n,
    "random_chance":random_chance,
    "ceiling_accuracy":round(ceiling_acc,2),"pipeline_accuracy":pipeline_acc,"baseline_accuracy":baseline_acc,
    "ceiling_compute_B":ceiling_compute,"pipeline_compute_B":pipeline_compute,"baseline_compute_B":baseline_compute,
    "ceiling_efficiency":round(ceiling_eff,4),"pipeline_efficiency":round(pipeline_eff,4),"baseline_efficiency":round(baseline_eff,4),
    "ceiling_wasted_votes":ceiling_wasted,"refiner_triggered":ref_triggered,"refiner_correct":ref_correct,
    "strategy_breakdown":strategy_stats,"pipeline_beats_ceiling":pipeline_acc >= ceiling_acc,
}
with open(CONFIG["angle1_file"],"w") as f: json.dump(angle1,f,indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (CommonsenseQA -- Three-Way Comparison)
  Random chance baseline: 20.0% (5 options A-E)

  Setup                                  |  Compute |  Accuracy |  Above Chance |   Acc/B
  ---------------------------------------+----------+-----------+---------------+--------
  Baseline  (1.5B x 5)                   |    7.5B  |     70.7% |        +50.7% |  9.427
  Pipeline  (3B+LoRA x1 + 1.5B x5)       |   10.5B  |     75.8% |        +55.8% |  7.219
  Ceiling   (3B base x 5)  <- THIS       |   15.0B  |     75.0% |        +55.0% |  5.000

  Pipeline vs Ceiling  : +0.8 pts  (pipeline uses 4.5B LESS)
  Ceiling  vs Baseline : +4.3 pts  (+7.5B more)
  Pipeline vs Baseline : +5.1 pts  (+3.0B more)

  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.
  Structured verbal planning outperforms raw model capacity on commonsense reasoning.

  Wasted votes -- Ceiling: 510/4500 (11.3%)
  Refiner: triggered 25, correct 7

  Strategy breakdown:
    majority          

In [14]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY + POSITION BIAS (A-E)
cons_scores = [r["vote_consistency"] for r in results]
mean_cons   = np.mean(cons_scores)

def bucket(scores):
    return {
        "all_wrong  (0%)"  : sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)" : sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)" : sum(1 for s in scores if s >= 0.8),
    }

dist      = bucket(cons_scores)
corr_cons = [r["vote_consistency"] for r in results if r["correct"]]

# Position bias: which letter does the 3B base model prefer?
from collections import defaultdict
letter_votes = defaultdict(int)
total_final_votes = 0
for r in results:
    for letter, count in r["vote_counts"].items():
        if letter in VALID_LETTERS:
            letter_votes[letter] += count
            total_final_votes    += count

# Known pipeline values from N=900 run
pipeline_cons = 0.757   # 75.7% from CSQA N=900
baseline_cons = 0.697   # 69.7% from CSQA N=900
pipeline_dist = {"all_wrong  (0%)":215,"low       (1-39%)":3,"medium  (40-79%)":2,"high   (80-100%)":680}
# Known pipeline position bias: A=22.3% B=20.9% C=19.1% D=23.8% E=14.0%
# Known baseline position bias: A=16.5% B=20.6% C=19.1% D=18.9% E=24.8% (E-bias)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (CommonsenseQA -- Three-Way)")
print("=" * 65)
print(f"  Mean correct-vote ratio (out of {CONFIG['n_votes']} per question):")
print(f"    Baseline (1.5B x5) : {baseline_cons*100:.1f}%  ({baseline_cons*5:.2f}/5 avg)")
print(f"    Pipeline (3B+LoRA) : {pipeline_cons*100:.1f}%  ({pipeline_cons*5:.2f}/5 avg)")
print(f"    Ceiling  (3B base) : {mean_cons*100:.1f}%  ({mean_cons*5:.2f}/5 avg)  <- THIS")

print(f"\n  Distribution (3B Ceiling vs Pipeline):")
print(f"  {'Bucket':<22} | {'Ceiling':>8} | {'Pipeline':>8}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}")
for bkt in ["all_wrong  (0%)","low       (1-39%)","medium  (40-79%)","high   (80-100%)"]:
    cv = dist[bkt]; pv = pipeline_dist.get(bkt,"—")
    print(f"  {bkt:<22} | {cv:>8} | {pv:>8}")

print(f"\n  Position Bias -- Option Letter Distribution:")
print(f"  Expected: 20.0% per option (uniform for 5-choice MCQ)")
print(f"  {'Letter':<6} | {'Ceiling':>8} | {'Pipeline':>8} | {'Baseline':>8} | {'Expected':>8}")
print(f"  {'-'*6}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}")
pipe_bias = {"A":22.3,"B":20.9,"C":19.1,"D":23.8,"E":14.0}
base_bias = {"A":16.5,"B":20.6,"C":19.1,"D":18.9,"E":24.8}
for letter in "ABCDE":
    ceil_pct = letter_votes.get(letter,0)/max(total_final_votes,1)*100
    print(f"  {letter:<6} | {ceil_pct:>7.1f}% | {pipe_bias.get(letter,0):>7.1f}% | {base_bias.get(letter,0):>7.1f}% | {'20.0%':>8}")

# Identify dominant bias
max_letter = max(letter_votes, key=lambda l: letter_votes.get(l,0))
max_pct    = letter_votes.get(max_letter,0)/max(total_final_votes,1)*100
if max_pct > 25:
    print(f"\n  ⚠ Position bias detected: ceiling model favours option {max_letter} ({max_pct:.1f}% vs 20% expected)")
else:
    print(f"\n  No strong position bias (max option: {max_letter} at {max_pct:.1f}%)")

if corr_cons:
    print(f"\n  Correct questions: ceiling consistency = {np.mean(corr_cons)*100:.1f}% (n={len(corr_cons)})")

angle2 = {
    "dataset":"CommonsenseQA","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_mean_consistency":round(mean_cons,4),
    "pipeline_mean_consistency":pipeline_cons,"baseline_mean_consistency":baseline_cons,
    "ceiling_distribution":dist,
    "ceiling_letter_dist":{l:round(letter_votes.get(l,0)/max(total_final_votes,1)*100,2) for l in "ABCDE"},
    "ceiling_correct_q_consistency":round(np.mean(corr_cons),4) if corr_cons else 0,
}
with open(CONFIG["angle2_file"],"w") as f: json.dump(angle2,f,indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY  (CommonsenseQA -- Three-Way)
  Mean correct-vote ratio (out of 5 per question):
    Baseline (1.5B x5) : 69.7%  (3.48/5 avg)
    Pipeline (3B+LoRA) : 75.7%  (3.79/5 avg)
    Ceiling  (3B base) : 73.1%  (3.65/5 avg)  <- THIS

  Distribution (3B Ceiling vs Pipeline):
  Bucket                 |  Ceiling | Pipeline
  -----------------------+----------+---------
  all_wrong  (0%)        |      114 |      215
  low       (1-39%)      |       75 |        3
  medium  (40-79%)       |      109 |        2
  high   (80-100%)       |      602 |      680

  Position Bias -- Option Letter Distribution:
  Expected: 20.0% per option (uniform for 5-choice MCQ)
  Letter |  Ceiling | Pipeline | Baseline | Expected
  -------+----------+----------+----------+---------
  A      |    18.2% |    22.3% |    16.5% |    20.0%
  B      |    18.9% |    20.9% |    20.6% |    20.0%
  C      |    20.7% |    19.1% |    19.1% |    20.0%
  D      |    23.2% |    23.8% |    18.9% |    20.0%
 

In [15]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(res, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total = len(res); ece = 0.0; calib_out = []
    false_conf = sum(1 for r in res if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in res if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |"); continue
        n=len(subset); acc=sum(r["correct"] for r in subset)/n; gap=abs(acc-mid)
        ece += (n/n_total)*gap; flag="Good" if gap<0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),"expected":mid,"gap":round(gap,4)})
    hc = [r for r in res if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc)/max(1,len(hc))*100
    print(f"  {'ECE (lower=better)':<26}   {ece:.4f}")
    print(f"  High-conf questions : {len(hc)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf}")
    return ece, calib_out, false_conf

# Known ECE from N=900 pipeline run
known_pipe_ece = 0.1413
known_base_ece = 0.1762

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (CommonsenseQA -- 3B Ceiling)")
print("=" * 65)
ceiling_ece, ceiling_calib, ceiling_false = calibration_report(results, "3B CEILING (this run)")
print(f"\n  ECE -- Three-Way:")
print(f"    Baseline (1.5B x5) : {known_base_ece:.4f}")
print(f"    Pipeline (3B+LoRA) : {known_pipe_ece:.4f}")
print(f"    Ceiling  (3B base) : {ceiling_ece:.4f}  <- THIS")
print(f"\n  Ceiling vs Pipeline: {ceiling_ece - known_pipe_ece:+.4f} ({'worse' if ceiling_ece > known_pipe_ece else 'better'})")
print(f"  False confidence (ceiling): {ceiling_false}")

angle3 = {
    "dataset":"CommonsenseQA","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_ece":round(ceiling_ece,4),"pipeline_ece":known_pipe_ece,"baseline_ece":known_base_ece,
    "ceiling_false_confidence":ceiling_false,"ceiling_calibration":ceiling_calib,
}
with open(CONFIG["angle3_file"],"w") as f: json.dump(angle3,f,indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (CommonsenseQA -- 3B Ceiling)

  [3B CEILING (this run)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   740 |     81.4% |       90% |  0.086 | Good
  High       (0.60-0.80)     |   124 |     47.6% |       70% |  0.224 | Poor
  Medium     (0.40-0.60)     |    29 |     37.9% |       50% |  0.121 | Good
  Low        (<0.40)         |     7 |     42.9% |       25% |  0.179 | Poor
  ECE (lower=better)           0.1073
  High-conf questions : 740  |  Accuracy when confident: 81.4%
  Confidently WRONG   : 138

  ECE -- Three-Way:
    Baseline (1.5B x5) : 0.1762
    Pipeline (3B+LoRA) : 0.1413
    Ceiling  (3B base) : 0.1073  <- THIS

  Ceiling vs Pipeline: -0.0340 (better)
  False confidence (ceiling): 138

Saved -> /kaggle/working/csqa_ceiling/angle3_confidence_calibration.json


In [16]:
# CELL 15 -- Full Three-Way Summary Table
with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

ca = a1["ceiling_accuracy"]
pa = a1["pipeline_accuracy"]
ba = a1["baseline_accuracy"]

print("=" * 75)
print("  CommonsenseQA -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING")
print(f"  N={a1['n_questions']} questions  |  Seed={CONFIG['random_seed']}  |  Qwen2.5 model family")
print(f"  Random chance: {CONFIG['random_chance']}% (5-option MCQ A-E)")
print("=" * 75)

rows = [
    ["Metric",                  "Baseline",          "Our Pipeline",          "3B Ceiling (this)"],
    ["Model",                   "Qwen2.5-1.5B x5",   "3B LoRA + 1.5B x5",    "Qwen2.5-3B base x5"],
    ["Fine-Tuning",             "None",              "LoRA on GSM8K",         "None"],
    ["Compute (param-passes)",  "7.5B",              "10.5B",                 "15.0B"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Overall Accuracy",        f"{ba:.1f}%",         f"{pa:.1f}%",            f"{ca:.1f}%"],
    ["Above Random Chance",     f"+{ba-20:.1f} pts",  f"+{pa-20:.1f} pts",     f"+{ca-20:.1f} pts"],
    ["vs Baseline",             "—",                 f"+{pa-ba:.1f} pts",      f"+{ca-ba:.1f} pts"],
    ["Pipeline vs Ceiling",     "—",                 f"{'WINS' if pa>=ca else 'loses'} {abs(pa-ca):.1f} pts","←"],
    ["Acc / Billion passes",    f"{ba/7.5:.3f}",      f"{pa/10.5:.3f}",        f"{ca/15.0:.3f}"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Vote Consistency",        f"{a2['baseline_mean_consistency']*100:.1f}%",
                                 f"{a2['pipeline_mean_consistency']*100:.1f}%",
                                 f"{a2['ceiling_mean_consistency']*100:.1f}%"],
    ["ECE (lower=better)",      f"{a3['baseline_ece']:.4f}",
                                 f"{a3['pipeline_ece']:.4f}",
                                 f"{a3['ceiling_ece']:.4f}"],
    ["False Confidence",        "—",                 "—",                     str(a3["ceiling_false_confidence"])],
]

col_w = [26, 20, 22, 20]
sep   = "-+-".join("-"*w for w in col_w)
for i, row in enumerate(rows):
    if "─────" in row[0]:
        print("  " + sep); continue
    line = " | ".join(str(c).ljust(col_w[j]) for j,c in enumerate(row))
    print("  " + line)
    if i == 0: print("  " + sep)

print()
print("  Position Bias (3B Ceiling):")
for letter, pct in sorted(a2["ceiling_letter_dist"].items()):
    bar = "█" * int(pct/2); flag = " ← bias" if pct > 25 else (" ← low" if pct < 15 else "")
    print(f"    {letter}: {pct:5.1f}%  {bar}{flag}")

print()
print("=" * 75)
if pa >= ca:
    print(f"  VERDICT: Pipeline BEATS the 3B ceiling (+{pa-ca:.1f} pts) at 30% lower cost.")
    print(f"  Fine-tuned math guide generalises to commonsense better than raw capacity.")
else:
    gap  = ca - pa
    frac = (pa - ba) / max(ca - ba, 0.01) * 100
    print(f"  VERDICT: 3B ceiling leads pipeline by {gap:.1f} pts.")
    print(f"  Pipeline delivers {frac:.0f}% of ceiling gain at {pa/ca*100:.0f}% of ceiling cost.")
    print(f"  The 3B base model has broad commonsense knowledge that scale can unlock.")
    print(f"  Pipeline's math-only fine-tuning cannot fully close the domain gap here.")
print("=" * 75)

full = {
    "dataset":"CommonsenseQA","seed":CONFIG["random_seed"],"n_questions":a1["n_questions"],
    "conditions":{
        "baseline":{"compute_B":7.5, "accuracy":ba,"model":"Qwen2.5-1.5B x5","fine_tuned":False},
        "pipeline":{"compute_B":10.5,"accuracy":pa,"model":"3B LoRA + 1.5B x5","fine_tuned":True},
        "ceiling" :{"compute_B":15.0,"accuracy":ca,"model":"3B base x5","fine_tuned":False},
    },
    "pipeline_beats_ceiling":pa >= ca,
    "angle1":a1,"angle2":a2,"angle3":a3,
}
with open(CONFIG["report_file"],"w") as f: json.dump(full,f,indent=2)
print(f"\nAll results saved to {OUTPUT_DIR}/")
print("Files: results.jsonl · eval_report.json · angle1/2/3.json")

  CommonsenseQA -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING
  N=900 questions  |  Seed=42  |  Qwen2.5 model family
  Random chance: 20.0% (5-option MCQ A-E)
  Metric                     | Baseline             | Our Pipeline           | 3B Ceiling (this)   
  ---------------------------+----------------------+------------------------+---------------------
  Model                      | Qwen2.5-1.5B x5      | 3B LoRA + 1.5B x5      | Qwen2.5-3B base x5  
  Fine-Tuning                | None                 | LoRA on GSM8K          | None                
  Compute (param-passes)     | 7.5B                 | 10.5B                  | 15.0B               
  ---------------------------+----------------------+------------------------+---------------------
  Overall Accuracy           | 70.7%                | 75.8%                  | 75.0%               
  Above Random Chance        | +50.7 pts            | +55.8 pts              | +55.0 pts           
  vs Baseline                | —        